# End-to-end non-imaging tutorial (laptop friendly)
This notebook walks through a minimal, single-node non-imaging analysis with the bundled response/data set in `tests/nitrates_resp_dir`. It mirrors the production pipeline without requiring a cluster.

**Workflow**
1. Load the lightweight response bundle and event data.
2. Build background and source models.
3. Scan a few on-source windows.
4. Report the best-fit amplitude, test statistic (sqrtTS), and basic source properties.

## 1) Imports and setup
We rely only on packages already listed in `requirements.txt`. If you followed the quickstart notebook, `NITRATES_RESP_DIR` is already set.

In [ ]:
import logging
import os
from pathlib import Path
import numpy as np
from astropy.io import fits
from nitrates.lib import (
    get_conn,
    get_info_tab,
    mask_detxy,
    convert_radec2thetaphi,
    convert_theta_phi2radec,
)
from nitrates.response import RayTraces
from nitrates.models import (
    Cutoff_Plaw_Flux,
    Source_Model_InOutFoV,
    CompoundModel,
    Sig_Bkg_Model,
    get_eflux_from_model,
)
from nitrates.llh_analysis import parse_bkg_csv, LLH_webins2, NLLH_ScipyMinimize_Wjacob

logging.basicConfig(level=logging.WARNING)

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
resp_dir = Path(os.environ.get('NITRATES_RESP_DIR', repo_root/'tests'/'nitrates_resp_dir')).resolve()
os.environ['NITRATES_RESP_DIR'] = str(resp_dir)
rt_dir = resp_dir/'ray_traces_detapp_npy'
solid_angle_dpi_path = resp_dir/'solid_angle_dpi.npy'
print(f"Using responses from: {resp_dir}")


## 2) Load event data and masks
We keep the exact screening used in the test suite: select good detectors, 14–500 keV photons, and a ±1000 s window around the trigger.

In [ ]:
# Trigger metadata
conn = get_conn(resp_dir/'results.db')
info_tab = get_info_tab(conn)
trig = info_tab['trigtimeMET'][0]

# Event data and detector mask
ev_data = fits.open(resp_dir/'filter_evdata.fits')[1].data
dmask = fits.open(resp_dir/'detmask.fits')[0].data
mask_vals = mask_detxy(dmask, ev_data)
bl_dmask = dmask == 0.0

# Basic time/energy cuts
t_start = trig - 1e3
t_end = trig + 1e3
bl_ev = (
    (ev_data['EVENT_FLAGS'] < 1)
    & (ev_data['ENERGY'] <= 500.0)
    & (ev_data['ENERGY'] >= 14.0)
    & (mask_vals == 0.0)
    & (ev_data['TIME'] <= t_end)
    & (ev_data['TIME'] >= t_start)
)
filtered_ev_data = ev_data[bl_ev]
print(f"Selected {len(filtered_ev_data)} events over {t_end - t_start:.1f} s")


## 3) Configure the spectral and spatial model
We adopt the cut-off power-law used in the paper and center the fit on the known GRB position (RA/Dec 233.117, -26.213). The `RayTraces` and response tables come from the lightweight bundle and are small enough for laptop use.

In [ ]:
# Energy bin edges (keV)
ebins0 = np.array([15.0, 24.0, 35.0, 48.0, 64.0])
ebins0 = np.append(ebins0, np.logspace(np.log10(84.0), np.log10(500.0), 5 + 1))[:-1]
ebins0 = np.round(ebins0, decimals=1)[:-1]
ebins1 = np.append(ebins0[1:], [350.0])

# Spacecraft attitude at trigger
attfile = fits.open(resp_dir/'attitude.fits')[1].data
att_ind = np.argmin(np.abs(attfile['TIME'] - trig))
att_quat = attfile['QPARAM'][att_ind]
source_ra_dec = (233.117, -26.213)
theta, phi = convert_radec2thetaphi(*source_ra_dec, att_quat)

# Flux model + responses
flux_params = {'A': 1.0, 'gamma': 0.5, 'Epeak': 1e2}
flux_mod = Cutoff_Plaw_Flux(E0=100.0)
rt_obj = RayTraces(rt_dir)
sig_mod = Source_Model_InOutFoV(
    flux_mod,
    [ebins0, ebins1],
    bl_dmask,
    rt_obj,
    use_deriv=True,
    resp_tab_dname=resp_dir/'resp_tabs_ebins',
    comp_flor_resp_dname=resp_dir/'comp_flor_resps',
    hp_flor_resp_dname=resp_dir/'hp_flor_resps',
)
sig_mod.set_theta_phi(theta, phi)
sig_mod.set_flux_params(flux_params)

print(f"Pointing quaternion index: {att_ind}")
print(f"Theta/Phi (deg): {theta:.3f}, {phi:.3f}")


## 4) Background model
The bundled `bkg_estimation.csv` already contains the diffuse and catalog sources. We reuse the helper in `parse_bkg_csv` to build a `CompoundModel`.

In [ ]:

bkg_df, bkg_name, _, bkg_mod, ps_mods = parse_bkg_csv(
    resp_dir/'bkg_estimation.csv',
    np.load(solid_angle_dpi_path),
    ebins0,
    ebins1,
    bl_dmask,
    rt_dir,
)

bkg_mod.has_deriv = False
bkg_mod_list = [bkg_mod]
for ps_mod in ps_mods:
    ps_mod.has_deriv = False
    bkg_mod_list.append(ps_mod)
if len(bkg_mod_list) > 1:
    bkg_mod = CompoundModel(bkg_mod_list)

bkg_row = bkg_df.iloc[np.argmin(np.abs(trig - bkg_df['time']))]
bkg_params = {pname: bkg_row[pname] for pname in bkg_mod.param_names}
print(f"Background components: {[m.name for m in bkg_mod_list]}")


## 5) Likelihood object
`LLH_webins2` handles the Poisson likelihood in energy and detector space. We only free the amplitude `A` of the transient while holding shape/location fixed for this tutorial.

In [ ]:
sig_pars = dict(flux_params)
sig_pars.update({'A': 1.0, 'theta': theta, 'phi': phi})

sig_bkg_mod = Sig_Bkg_Model(bl_dmask, sig_mod, bkg_mod, use_deriv=True)
sig_bkg_mod.set_bkg_params(bkg_params)
sig_bkg_mod.set_sig_params(sig_pars)

sig_llh_obj = LLH_webins2(filtered_ev_data, ebins0, ebins1, bl_dmask, has_err=True)
sig_llh_obj.set_model(sig_bkg_mod)

sig_miner = NLLH_ScipyMinimize_Wjacob('')
sig_miner.set_llh(sig_llh_obj)
sig_miner.set_fixed_params(['A'], fixed=False)


## 6) Scan a few on-source windows
To keep runtime short, we test three durations centered on the trigger. The helper below returns the best-fit amplitude and sqrtTS for each window.

In [ ]:
negligible_amplitude = 1e-10  # amplitude effectively zero for background-only comparison
def evaluate_window(t0, duration):
    t1 = t0 + duration
    sig_llh_obj.set_time(t0, t1)
    bf_vals, nllh, _ = sig_miner.minimize()
    amp = bf_vals[0][0] if isinstance(bf_vals[0], (list, tuple, np.ndarray)) else bf_vals[0]
    bkg_nllh = -sig_llh_obj.get_logprob({'A': negligible_amplitude})
    sqrtTS = np.sqrt(2.0 * (bkg_nllh - nllh[0]))
    return {'t0': t0, 't1': t1, 'duration': duration, 'amplitude': amp, 'sqrtTS': sqrtTS}

windows = [0.512, 1.024, 2.048]
results = [evaluate_window(trig - 0.5 * dur, dur) for dur in windows]
results


## 7) Summarize the best candidate
Pick the highest TS window, convert back to sky coordinates, and estimate the 15–150 keV energy flux using the best-fit amplitude.

In [ ]:
best = max(results, key=lambda row: row['sqrtTS'])
ra_best, dec_best = convert_theta_phi2radec(theta, phi, att_quat)
fluence_band = (15.0, 150.0)
fluence = get_eflux_from_model(flux_mod, {'A': best['amplitude'], 'gamma': flux_params['gamma'], 'Epeak': flux_params['Epeak']}, *fluence_band)

print(f"Best window: {best['duration']:.3f}s starting at {best['t0'] - trig:+.3f} s relative to trigger")
print(f"sqrtTS: {best['sqrtTS']:.2f}")
print(f"Amplitude A: {best['amplitude']:.3e}")
print(f"Sky position (RA, Dec): ({ra_best:.3f}, {dec_best:.3f})")
print(f"Energy flux {fluence_band[0]:.0f}-{fluence_band[1]:.0f} keV: {fluence:.3e} erg/cm^2/s")


You now have a minimal, reproducible non-imaging analysis that runs end-to-end on a laptop. Adjust the source coordinates, durations, or spectral parameters to explore alternative candidates or rapid follow-up scenarios.